# Module 1: Building a Multi-Agent Customer Service System

## Overview

In this module, you'll build a production-ready multi-agent system for e-commerce customer service using the Strands Agent SDK. The system uses a **cost-optimized architecture** with:

- **Orchestrator Agent** (Claude Sonnet 4.5): Handles intent classification and complex reasoning
- **Order Agent** (Claude Haiku 4.5): Handles order inquiries, tracking, returns
- **Product Agent** (Claude Haiku 4.5): Handles product search, recommendations, inventory
- **Account Agent** (Claude Haiku 4.5): Handles account management, passwords, preferences

### Learning Objectives
1. Build specialized agents with Strands SDK and real tool implementations
2. Create an orchestrator for intelligent request routing
3. Understand cost optimization with mixed LLM strategy
4. Test the multi-agent system with various queries

### Time: ~60 minutes

## Step 1: Environment Setup

First, let's set up our environment and verify we have access to the required resources.

In [2]:
# Install dependencies if needed
!pip install -q strands-agents strands-agents-tools boto3


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [3]:
import boto3
import json
import os
import sys
from datetime import datetime

# Get AWS region
session = boto3.Session()
REGION = session.region_name or 'us-west-2'
print(f"AWS Region: {REGION}")

# Set environment variables for tools
os.environ['AWS_REGION'] = REGION

# Get table names from SSM (set by workshop infrastructure)
ssm = boto3.client('ssm', region_name=REGION)

try:
    os.environ['ORDERS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-orders-table')['Parameter']['Value']
    os.environ['ACCOUNTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-accounts-table')['Parameter']['Value']
    os.environ['PRODUCTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-products-table')['Parameter']['Value']
    print(f"✓ Orders Table: {os.environ['ORDERS_TABLE_NAME']}")
    print(f"✓ Accounts Table: {os.environ['ACCOUNTS_TABLE_NAME']}")
    print(f"✓ Products Table: {os.environ['PRODUCTS_TABLE_NAME']}")
except Exception as e:
    print(f"Note: Could not retrieve SSM parameters ({e})")
    print("Using default table names - ensure infrastructure is deployed")
    os.environ['ORDERS_TABLE_NAME'] = 'ecommerce-workshop-orders'
    os.environ['ACCOUNTS_TABLE_NAME'] = 'ecommerce-workshop-accounts'
    os.environ['PRODUCTS_TABLE_NAME'] = 'ecommerce-workshop-products'

AWS Region: us-west-2
✓ Orders Table: ecommerce-workshop-orders
✓ Accounts Table: ecommerce-workshop-accounts
✓ Products Table: ecommerce-workshop-products


## Step 2: Explore the Tools

Before building agents, let's understand the tools they'll use. Each tool is implemented as a function that interacts with real AWS services.

In [4]:
# Add tools and agents to path
sys.path.insert(0, 'tools')
sys.path.insert(0, 'agents')

# Import tools
from order_tools import get_order_status, track_shipment, process_return, get_customer_orders
from product_tools import search_products, check_inventory, get_product_recommendations
from account_tools import get_account_info, get_membership_benefits, initiate_password_reset

### 2.1 Test Order Tools

These tools interact with DynamoDB to manage orders.

In [5]:
# Test get_order_status
result = get_order_status(order_id="ORD-2024-10002")
print("Order Status Result:")
print(json.dumps(result, indent=2, default=str))

Order Status Result:
{
  "success": true,
  "order_id": "ORD-2024-10002",
  "status": "shipped",
  "order_date": "2024-12-28",
  "customer_email": "sarah.johnson@email.com",
  "items": [
    {
      "name": "Smart Watch Pro",
      "quantity": "1",
      "price": "299.99",
      "product_id": "PROD-015"
    },
    {
      "name": "Watch Band - Leather",
      "quantity": "2",
      "price": "29.99",
      "product_id": "PROD-016"
    }
  ],
  "total": 359.97,
  "shipping_address": {
    "zip": "90001",
    "city": "Los Angeles",
    "state": "CA"
  },
  "shipping_carrier": "UPS",
  "tracking_number": "1Z999AA10123456784",
  "estimated_delivery": "2025-01-08"
}


In [6]:
# Test track_shipment
result = track_shipment(order_id="ORD-2024-10002")
print("Tracking Result:")
print(json.dumps(result, indent=2, default=str))

Tracking Result:
{
  "success": true,
  "order_id": "ORD-2024-10002",
  "status": "shipped",
  "shipping_carrier": "UPS",
  "tracking_number": "1Z999AA10123456784",
  "estimated_delivery": "2025-01-08",
  "tracking_url": "https://www.ups.com/track?tracknum=1Z999AA10123456784"
}


### 2.2 Test Product Tools

These tools interact with Bedrock Knowledge Base for product information.

In [7]:
# Test search_products
result = search_products(query="wireless headphones with noise cancellation")
print("Product Search Result:")
print(json.dumps(result, indent=2, default=str))

Product Search Result:
{
  "success": true,
  "query": "wireless headphones with noise cancellation",
  "category_filter": null,
  "result_count": 5,
  "results": [
    {
      "product_id": "PROD-088",
      "name": "Webcam 4K HDR",
      "price": 199.99,
      "category": "Cameras",
      "in_stock": false,
      "description": "Professional 4K webcam with HDR, auto-focus, dual microphones, and privacy shutter."
    },
    {
      "product_id": "PROD-055",
      "name": "Noise Canceling Earbuds",
      "price": 149.99,
      "category": "Audio",
      "in_stock": true,
      "description": "True wireless earbuds with adaptive noise cancellation, transparency mode, and 8-hour battery life per charge."
    },
    {
      "product_id": "PROD-077",
      "name": "Gaming Mechanical Keyboard RGB",
      "price": 159.99,
      "category": "Gaming",
      "in_stock": true,
      "description": "Full-size mechanical gaming keyboard with hot-swappable switches, per-key RGB lighting, and detach

In [8]:
# Test check_inventory
result = check_inventory(product_id="PROD-001")
print("Inventory Check Result:")
print(json.dumps(result, indent=2, default=str))

Inventory Check Result:
{
  "success": true,
  "product_id": "PROD-001",
  "in_stock": true,
  "quantity_available": 150.0,
  "message": "In stock and ready to ship"
}


### 2.3 Test Account Tools

These tools manage customer accounts in DynamoDB.

In [9]:
# Test get_account_info
result = get_account_info(customer_email="sarah.johnson@email.com")
print("Account Info Result:")
print(json.dumps(result, indent=2, default=str))

Account Info Result:
{
  "success": true,
  "customer_id": "CUST-1002",
  "email": "sarah.johnson@email.com",
  "first_name": "Sarah",
  "last_name": "Johnson",
  "phone": "+1-555-0102",
  "account_status": "active",
  "membership_tier": "platinum",
  "member_since": "2021-08-22",
  "total_orders": "42",
  "total_spent": 8934.56,
  "preferences": {
    "marketing_emails": false,
    "email_notifications": true,
    "sms_notifications": true
  },
  "default_shipping_address": {
    "zip": "90001",
    "country": "USA",
    "state": "CA",
    "city": "Los Angeles",
    "street": "456 Oak Avenue"
  }
}


In [10]:
# Test get_membership_benefits
result = get_membership_benefits(membership_tier="platinum")
print("Membership Benefits:")
print(json.dumps(result, indent=2, default=str))

Membership Benefits:
{
  "success": true,
  "tier": "Platinum",
  "free_shipping_threshold": 0,
  "return_window_days": 60,
  "exclusive_sales": true,
  "priority_support": true,
  "points_multiplier": 2.0,
  "benefits": [
    "Free shipping on all orders",
    "60-day return window",
    "Priority customer support",
    "Exclusive member-only sales",
    "Earn 2 points per $1 spent",
    "Birthday discount (25% off)",
    "Free gift wrapping"
  ]
}


## Step 3: Build Specialized Agents

Now let's create our specialized agents using Claude Haiku 4.5 for cost efficiency.

In [11]:
from strands import Agent
from strands.models import BedrockModel

# Model configuration using global cross-region inference profiles
HAIKU_MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET_MODEL_ID = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Pricing per 1M tokens (approximate, check AWS pricing page for latest)
# https://aws.amazon.com/bedrock/pricing/
HAIKU_INPUT_PRICE = 0.80    # $0.80 per 1M input tokens
HAIKU_OUTPUT_PRICE = 4.00   # $4.00 per 1M output tokens
SONNET_INPUT_PRICE = 3.00   # $3.00 per 1M input tokens  
SONNET_OUTPUT_PRICE = 15.00 # $15.00 per 1M output tokens

print(f"Haiku Model: {HAIKU_MODEL_ID}")
print(f"Sonnet Model: {SONNET_MODEL_ID}")

Haiku Model: global.anthropic.claude-haiku-4-5-20251001-v1:0
Sonnet Model: global.anthropic.claude-sonnet-4-5-20250929-v1:0


### 3.1 Create Order Agent

In [12]:
from order_agent import create_order_agent

order_agent = create_order_agent(region=REGION)
print(f"Order Agent created with model: {HAIKU_MODEL_ID}")

Order Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0


In [13]:
# Test Order Agent directly
response = order_agent("What's the status of order ORD-2024-10002?")
print("Order Agent Response:")
print(response)


Tool #1: get_order_status
Great! Here's the status of your order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number provided above. Would you like me to get more detailed tracking information or help you with anything else?Order Agent Response:
Great! Here's the status of your order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier

### 3.2 Create Product Agent

In [14]:
from product_agent import create_product_agent

product_agent = create_product_agent(region=REGION)
print(f"Product Agent created with model: {HAIKU_MODEL_ID}")

Product Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0


In [15]:
# Test Product Agent directly
response = product_agent("Do you have any wireless headphones under $100?")
print("Product Agent Response:")
print(response)

I'll search for wireless headphones under $100 for you.
Tool #1: search_products
Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones - $79.99** (PROD-001)
- **Type:** Over-ear headphones
- **Key Features:**
  - Active noise cancellation
  - 40-hour battery life
  - Comfortable memory foam ear cushions
  - Premium quality
- **Availability:** In stock ✓

I also found another option that's slightly above your budget:

**Noise Canceling Earbuds - $149.99** (PROD-055)
- **Type:** True wireless earbuds
- **Key Features:**
  - Adaptive noise cancellation
  - Transparency mode
  - 8-hour battery life per charge
- **Availability:** In stock ✓

The **Wireless Bluetooth Headphones at $79.99** is perfect if you're looking to stay under $100! Would you like more details about this product, or would you like me to check availability or get recommendations for complementary accessories?Product Agent Response:
Great! I found one wireless headphone option under $10

### 3.3 Create Account Agent

In [16]:
from account_agent import create_account_agent

account_agent = create_account_agent(region=REGION)
print(f"Account Agent created with model: {HAIKU_MODEL_ID}")

Account Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0


In [17]:
# Test Account Agent directly
response = account_agent("What are the benefits of Gold membership?")
print("Account Agent Response:")
print(response)


Tool #1: get_membership_benefits
Great question! Here are the benefits of **Gold membership**:

✨ **Gold Membership Benefits:**
- **Free Shipping** on orders over $25 (compared to $50 for Standard)
- **45-Day Return Window** for returns and exchanges
- **Early Access to Sales** - get first dibs on exclusive deals
- **Earn 1.5 Points per $1 Spent** - accelerated rewards earning
- **Birthday Discount** - 15% off during your birthday month

**Gold vs. Standard:**
Gold membership is a great middle-tier option that gives you better shipping thresholds and exclusive sale access compared to our Standard tier.

**If you're interested in even more benefits**, our **Platinum tier** includes:
- Free shipping on *all* orders (no minimum)
- Priority customer support
- Plus all the Gold benefits

Would you like to know more about Platinum membership, or do you have any other questions about Gold benefits?Account Agent Response:
Great question! Here are the benefits of **Gold membership**:

✨ **Gold

## Step 4: Create the Orchestrator

The Orchestrator uses Claude Sonnet 4.5 (larger model) for complex reasoning and routes requests to specialized agents.

In [18]:
from orchestrator import MultiAgentCustomerService

# Create the multi-agent system
customer_service = MultiAgentCustomerService(region=REGION)
print("Multi-Agent Customer Service System initialized!")
print(f"- Orchestrator: {SONNET_MODEL_ID}")
print(f"- Sub-agents: {HAIKU_MODEL_ID}")

Multi-Agent Customer Service System initialized!
- Orchestrator: global.anthropic.claude-sonnet-4-5-20250929-v1:0
- Sub-agents: global.anthropic.claude-haiku-4-5-20251001-v1:0


## Step 5: Test the Multi-Agent System

Let's test various customer scenarios to see how the orchestrator routes requests.

In [19]:
# Test Case 1: Order inquiry (should route to Order Agent)
print("="*60)
print("Test Case 1: Order Inquiry")
print("="*60)

response = customer_service.chat("Where is my order ORD-2024-10002?")
print(f"Customer: Where is my order ORD-2024-10002?")
print(f"\nAgent: {response}")

Test Case 1: Order Inquiry

Tool #1: delegate_to_order_agent
I'll check the status and tracking information for your order.
Tool #1: get_order_status

Tool #2: track_shipment
Great news! Your order **ORD-2024-10002** is on its way! 📦

**Order Status:** Shipped

**Shipping Details:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** January 8, 2025
- **Shipping To:** Los Angeles, CA 90001

**Items in Your Order:**
- 1x Smart Watch Pro ($299.99)
- 2x Watch Band - Leather ($29.99 each)

**Total:** $359.97

You can track your shipment in real-time using your UPS tracking number here: https://www.ups.com/track?tracknum=1Z999AA10123456784

Is there anything else you'd like to know about your order?Great news! Your order **ORD-2024-10002** is on its way! 📦

**Order Status:** Shipped

**Shipping Details:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** January 8, 2025
- **Shipping To:** Los Angeles, CA 90001

**Items 

In [20]:
# Test Case 2: Product search (should route to Product Agent)
print("="*60)
print("Test Case 2: Product Search")
print("="*60)

response = customer_service.chat("I'm looking for a good gaming keyboard with RGB lighting")
print(f"Customer: I'm looking for a good gaming keyboard with RGB lighting")
print(f"\nAgent: {response}")

Test Case 2: Product Search

Tool #2: delegate_to_product_agent
I'll search for gaming keyboards with RGB lighting for you.
Tool #1: search_products
Great! I found a gaming keyboard with RGB lighting. Let me get more detailed information about it.
Tool #2: get_product_details
Perfect! Here's what I found for you:

## **Gaming Mechanical Keyboard RGB** (PROD-077)
**Price:** $159.99

### Key Features:
- **Mechanical Switches:** Hot-swappable Cherry MX compatible switches (easily swap switches without soldering)
- **RGB Lighting:** Per-key RGB with 16.8 million colors for full customization
- **Connection:** USB-C detachable cable for easy portability
- **Anti-Ghosting:** Full N-key rollover for responsive gaming
- **Media Controls:** Dedicated media keys + volume wheel
- **Wrist Rest:** Detachable wrist rest for comfort during long gaming sessions
- **Layout:** Full-size keyboard

### Additional Info:
- **Rating:** 4.6 out of 5 stars
- **Stock:** 60 units in stock
- **Warranty:** 2 years

In [21]:
# Test Case 3: Account management (should route to Account Agent)
print("="*60)
print("Test Case 3: Account Management")
print("="*60)

response = customer_service.chat("I need to reset my password for john.smith@email.com")
print(f"Customer: I need to reset my password for john.smith@email.com")
print(f"\nAgent: {response}")

Test Case 3: Account Management

Tool #3: delegate_to_account_agent
I'll initiate a password reset for your account right away.
Tool #1: initiate_password_reset
Perfect! I've initiated the password reset process for john.smith@email.com. Here's what you need to know:

**Password Reset Details:**
- ✓ A password reset link has been sent to john.smith@email.com
- **Check your email** (including your spam/junk folder, just in case)
- The reset link will **expire in 1 hour**, so please complete the reset promptly
- Follow the link in the email to create your new password

**Important Security Notes:**
- Never share your password reset link with anyone
- I cannot share reset links through chat for security reasons
- Make sure to use a strong, unique password for your new account

If you don't receive the email within a few minutes, please:
1. Check your spam/junk folder
2. Verify the email address is correct
3. Contact our support team if you continue to have issues

Is there anything else I

In [22]:
# Test Case 4: Complex multi-domain query (should use multiple agents)
print("="*60)
print("Test Case 4: Complex Multi-Domain Query")
print("="*60)

response = customer_service.chat(
    "I want to return order ORD-2024-10001 because the headphones don't fit well. "
    "Can you recommend a different pair that might be more comfortable?"
)
print(f"Customer: I want to return order ORD-2024-10001 because the headphones don't fit well. Can you recommend a different pair that might be more comfortable?")
print(f"\nAgent: {response}")

Test Case 4: Complex Multi-Domain Query

Tool #4: delegate_to_order_agent
I'll process a return request for your order ORD-2024-10001.
Tool #3: process_return
I'm sorry, but I'm unable to process a return for **ORD-2024-10001** at this time. 

**Reason:** The return window has expired. This order was placed 390 days ago, and we only accept returns within 30 days of purchase.

**What you can do:**
Since this is outside our standard return policy, I'd recommend contacting our customer service team directly. They may be able to discuss alternative solutions with you, such as:
- Warranty coverage (if applicable)
- Replacement options
- Other accommodations

Would you like me to help you with anything else, or do you have another order you'd like assistance with?
Tool #5: delegate_to_product_agent
I'll get some personalized recommendations for comfortable headphones based on your fit issues with your previous pair.
Tool #3: get_product_recommendations
Great! I found some recommendations. Le

In [23]:
# Test Case 5: Inventory check
print("="*60)
print("Test Case 5: Inventory Check")
print("="*60)

response = customer_service.chat("Is the 4K webcam PROD-088 in stock?")
print(f"Customer: Is the 4K webcam PROD-088 in stock?")
print(f"\nAgent: {response}")

Test Case 5: Inventory Check

Tool #6: delegate_to_product_agent

Tool #6: check_inventory
Unfortunately, **PROD-088 is currently out of stock**. 

**Restock Information:**
- **Expected Restock Date:** January 15, 2025
- **Current Stock:** 0 units available

Would you like me to:
1. **Search for alternative 4K webcams** that are currently in stock?
2. **Get product details** about PROD-088 so you know what to expect when it's back in stock?
3. **Recommend similar webcams** with comparable features?

Let me know how I can help!Unfortunately, **PROD-088 is currently out of stock**. 

**Restock Information:**
- **Expected Restock Date:** January 15, 2025
- **Current Stock:** 0 units available

Would you like me to:
1. **Search for alternative 4K webcams** that are currently in stock?
2. **Get product details** about PROD-088 so you know what to expect when it's back in stock?
3. **Recommend similar webcams** with comparable features?

Let me know how I can help!Customer: Is the 4K webcam 

## Step 6: Cost Analysis

Let's analyze the cost difference between using a single large model vs. our mixed architecture.

In [24]:
# Simulated token usage for a typical customer service session
# Based on average query complexity

def calculate_cost_comparison():
    """Compare cost between all-Sonnet vs mixed architecture"""
    
    # Average tokens per interaction (approximate)
    orchestrator_input = 800   # System prompt + query + tool descriptions
    orchestrator_output = 200  # Routing decision + brief response
    sub_agent_input = 600      # System prompt + delegated query + tools
    sub_agent_output = 400     # Detailed response
    
    # Scenario: 1000 customer interactions
    num_interactions = 1000
    
    # All-Sonnet Architecture (single large model)
    all_sonnet_input = (orchestrator_input + sub_agent_input) * num_interactions
    all_sonnet_output = (orchestrator_output + sub_agent_output) * num_interactions
    all_sonnet_cost = (
        (all_sonnet_input / 1_000_000) * SONNET_INPUT_PRICE +
        (all_sonnet_output / 1_000_000) * SONNET_OUTPUT_PRICE
    )
    
    # Mixed Architecture (Sonnet orchestrator + Haiku sub-agents)
    mixed_orchestrator_input = orchestrator_input * num_interactions
    mixed_orchestrator_output = orchestrator_output * num_interactions
    mixed_subagent_input = sub_agent_input * num_interactions
    mixed_subagent_output = sub_agent_output * num_interactions
    
    mixed_cost = (
        # Orchestrator (Sonnet 4.5)
        (mixed_orchestrator_input / 1_000_000) * SONNET_INPUT_PRICE +
        (mixed_orchestrator_output / 1_000_000) * SONNET_OUTPUT_PRICE +
        # Sub-agents (Haiku 4.5)
        (mixed_subagent_input / 1_000_000) * HAIKU_INPUT_PRICE +
        (mixed_subagent_output / 1_000_000) * HAIKU_OUTPUT_PRICE
    )
    
    savings = all_sonnet_cost - mixed_cost
    savings_percent = (savings / all_sonnet_cost) * 100
    
    return {
        'num_interactions': num_interactions,
        'all_sonnet_cost': round(all_sonnet_cost, 2),
        'mixed_cost': round(mixed_cost, 2),
        'savings': round(savings, 2),
        'savings_percent': round(savings_percent, 1)
    }

cost_analysis = calculate_cost_comparison()

print("Cost Comparison Analysis")
print("="*50)
print(f"Scenario: {cost_analysis['num_interactions']:,} customer interactions")
print()
print(f"All-Sonnet Architecture:   ${cost_analysis['all_sonnet_cost']:.2f}")
print(f"Mixed Architecture:        ${cost_analysis['mixed_cost']:.2f}")
print()
print(f"Savings:                   ${cost_analysis['savings']:.2f} ({cost_analysis['savings_percent']}%)")
print()
print("Note: Mixed architecture uses Sonnet 4.5 only for orchestration,")
print("      while Haiku 4.5 handles specialized tasks.")

Cost Comparison Analysis
Scenario: 1,000 customer interactions

All-Sonnet Architecture:   $13.20
Mixed Architecture:        $7.48

Savings:                   $5.72 (43.3%)

Note: Mixed architecture uses Sonnet 4.5 only for orchestration,
      while Haiku 4.5 handles specialized tasks.


## Step 7: Usage Statistics

Review how the system distributed requests across agents.

In [25]:
# Get usage statistics from our session
stats = customer_service.get_usage_stats()

print("Session Usage Statistics")
print("="*40)
print(f"Total Requests:      {stats['total_requests']}")
print(f"Orchestrator Calls:  {stats['orchestrator_calls']}")
print()
print("The orchestrator intelligently routes requests to")
print("specialized agents based on query intent.")

Session Usage Statistics
Total Requests:      5
Orchestrator Calls:  5

The orchestrator intelligently routes requests to
specialized agents based on query intent.


## Summary

In this module, you built a multi-agent customer service system that:

1. **Uses real tools** that interact with DynamoDB and Bedrock Knowledge Base
2. **Optimizes costs** by using smaller models for specialized tasks
3. **Routes intelligently** using a larger model for orchestration only
4. **Handles complex queries** that span multiple domains

### Key Takeaways

- **Cost Optimization**: Using Haiku 4.5 for sub-agents instead of Sonnet 4.5 everywhere can save ~50% on LLM costs
- **Specialization**: Each agent has focused tools and prompts for its domain
- **Orchestration**: The orchestrator handles intent classification and complex reasoning
- **Scalability**: This architecture scales well as you add more specialized agents

### Next Steps

In **Module 2**, we'll evaluate this system with a comprehensive test dataset and establish baseline metrics for production monitoring.

In [26]:
# Save region for use in next module
# Note: customer_service object cannot be pickled due to boto3 clients
# Module 2 will recreate it from REGION
%store REGION
print("Session data saved for Module 2!")
print("Module 2 will recreate the customer_service instance from the region.")

Stored 'REGION' (str)
Session data saved for Module 2!
Module 2 will recreate the customer_service instance from the region.
